# COLMAP PatchMatch Multi-View Stereo (MVS) on Google Colab (T4 GPU)
This notebook runs GPU-accelerated **PatchMatch Multi-View Stereo** and **Stereo Fusion** using the registered camera poses and keyframes from your Stage 02 SfM pipeline.

### Step 1: Verify NVIDIA T4 GPU Environment

In [ ]:
!nvidia-smi

### Step 2: Install CUDA-Enabled PyCOLMAP and Dependencies

In [ ]:
# Install official CUDA 12 PyCOLMAP wheel
!pip uninstall -y pycolmap
!pip install --upgrade pycolmap-cuda12 open3d ultralytics trimesh

import pycolmap, torch
print('PyTorch CUDA available:', torch.cuda.is_available())
print('PyCOLMAP has CUDA:', pycolmap.has_cuda)
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

### Step 3: Automatically Fetch and Extract the MVS Dataset (No Manual Upload Needed!)

In [ ]:
import os, zipfile, shutil

# Fetch bundle directly from repository if not present
if not os.path.exists('colab_mvs_input.zip'):
    if not os.path.exists('vision-lab'):
        !git clone https://github.com/Abhi35968/vision-lab.git
    shutil.copy('vision-lab/scripts/colab_mvs_input.zip', 'colab_mvs_input.zip')

# Extract bundle
with zipfile.ZipFile('colab_mvs_input.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print('Extraction complete! Files ready for PatchMatch MVS:')
!ls -la

### Step 4: Execute CUDA PatchMatch Stereo & Fusion (~1 to 2 minutes)

In [ ]:
!python colab_patchmatch_mvs.py --sparse sparse --images keyframes --workspace mvs_output

### Step 5: Inspect and Download Reconstructed 3D Assets

In [ ]:
from google.colab import files
import json

with open('mvs_output/mvs_summary.json') as f:
    summary = json.load(f)
print('MVS Results Summary:')
print(json.dumps(summary, indent=2))

# Download the final MVS dense point cloud and mesh
files.download('mvs_output/mvs/fused.ply')
files.download('mvs_output/mvs/mvs_poisson_mesh.ply')